# Module 5 - Class 4: t-SNE and UMAP Visualization

**Dataset:** Mall Customer Segmentation  
**Objective:** Compare PCA, t-SNE, and UMAP for 2D visualization of customer clusters.

### What you will learn
- PCA limitations for non-linear structure
- t-SNE: preserves local neighborhood structure
- UMAP: preserves both local and global structure
- Side-by-side visual comparison

---

## 0. Setup

In [ ]:
!pip install umap-learn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import umap
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")

## 1. Load and Standardize Data

In [ ]:
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/Mall_Customers.csv"
df = pd.read_csv(url)

df['Gender_encoded'] = (df['Gender'] == 'Male').astype(int)
features = ['Gender_encoded', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Shape: {X_scaled.shape}")
print(f"Features: {features}")

## 2. K-Means Cluster Labels (for coloring)

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

print(f"Cluster distribution:")
print(pd.Series(cluster_labels).value_counts().sort_index())

## 3. PCA to 2D

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"PCA explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%")

## 4. t-SNE to 2D

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

print(f"t-SNE complete. KL divergence: {tsne.kl_divergence_:.4f}")

## 5. UMAP to 2D

In [ ]:
reducer = umap.UMAP(n_components=2, n_neighbors=15, random_state=42)
X_umap = reducer.fit_transform(X_scaled)

print("UMAP complete.")

## 6. Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
palette = sns.color_palette('Set2', 5)

# PCA
for c in range(5):
    mask = cluster_labels == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[palette[c]], label=f'Cluster {c}', alpha=0.7, s=40)
axes[0].set_title('PCA', fontsize=14, fontweight='bold')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# t-SNE
for c in range(5):
    mask = cluster_labels == c
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=[palette[c]], label=f'Cluster {c}', alpha=0.7, s=40)
axes[1].set_title('t-SNE (perplexity=30)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# UMAP
for c in range(5):
    mask = cluster_labels == c
    axes[2].scatter(X_umap[mask, 0], X_umap[mask, 1], c=[palette[c]], label=f'Cluster {c}', alpha=0.7, s=40)
axes[2].set_title('UMAP (n_neighbors=15)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('UMAP 1')
axes[2].set_ylabel('UMAP 2')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Dimensionality Reduction Comparison — Mall Customers (colored by K-Means clusters)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Quick Reference: PCA vs t-SNE vs UMAP

| Method | Type | Preserves | Speed | Axes Interpretable? |
|--------|------|-----------|-------|--------------------|
| PCA | Linear | Global variance | Fast | Yes (loadings) |
| t-SNE | Non-linear | Local neighborhoods | Slow | No |
| UMAP | Non-linear | Local + global structure | Medium | No |

## 7. TODO: Which Visualization Best Reveals Cluster Structure?

Look at the 3 plots above and answer:

1. Which method produces the most visually separated clusters?
2. Why does PCA sometimes fail to separate clusters that t-SNE/UMAP can?
3. Would you use t-SNE or UMAP for a dataset with 1 million rows? Why?
4. Why should you NOT interpret distances in t-SNE plots the same way as in PCA plots?

**TODO: Your answer here**

*Write your answer in this cell.*


---
## Summary

| Concept | Details |
|---------|--------|
| PCA | Linear projection onto max-variance directions. Fast, interpretable, but misses non-linear structure |
| t-SNE | Preserves local neighborhoods via probability distributions. Good for visualization, slow on large data |
| UMAP | Topological approach — preserves local AND global structure. Faster than t-SNE, scales better |
| perplexity (t-SNE) | Controls neighborhood size. Typical range: 5-50 |
| n_neighbors (UMAP) | Controls local vs global balance. Low = local detail, high = global structure |
| Key rule | t-SNE/UMAP are for visualization only — do not use the 2D output for clustering or modeling |